# PFF World Cup 2022 as a second training / validation set

Same pipeline as `possession_lgbm.ipynb` (v1 features, LightGBM, Viterbi, boundary-aware decoder),
now with **64 more matches** from a different provider and a different possession definition.

**Why PFF.** IDSSE's owner flag changes hands ~2x as often as Metrica's public window. PFF's labels
are derived from its *event* stream (kloppy's rule: `OUT`/`END` → dead, kickoffs/`OTB` → alive,
`home_ball` → owner, carried forward), so possession only changes at a recorded on-ball event —
closer to "controlled possession". Two datasets with two definitions also let us prefer what is
**robust** rather than what fits one provider's quirks.

**Get the data** (free, behind a form — has to be you):
1. https://www.blog.fc.pff.com/blog/enhanced-2022-world-cup-dataset → fill the form → download.
2. Unpack under `data/external/pff/` (gitignored). Any layout works if tracking files end in
   `.jsonl.bz2` and metadata / rosters sit in folders whose names contain `meta` / `roster`.

**Rules.** Labels on other games are allowed in any amount (RULES.md); the technical document must
say: PFF FC / Gradient Sports FIFA World Cup 2022 open data, labels derived algorithmically from
PFF's analyst-collected event stream, 64 matches. The public BAR-ATH window is used once, at the end,
as an alignment check — never to choose.

Run `possession_lgbm.ipynb` §3 first: it writes the BAR-ATH-calibrated noise this notebook reuses.

In [ ]:
import os, sys, json, time, subprocess
ROOT = os.path.abspath('..') if os.path.isfile(os.path.join('..', 'games.py')) else os.path.abspath('.')
for p in (ROOT, os.path.join(ROOT, 'starter'), os.path.join(ROOT, 'evaluation')):
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from solution import (atd, boundary, container as C, dataset_fit, degrade, evaluate_external as ev,
                      features, idsse, labels, pff, submission)
pm = C.pm

TEAM = 'MyTeam_pff'          # submission file name for this variant
SEED = 0
FOLDS = 8                    # PFF matches are grouped into 8 folds (64 x leave-one-out is too slow)
PFF_STRIDE = 15              # every 15th PFF frame, one orientation -- RAM
FEATURE_FN = features.build_features_v1
TRAIN_CFG = {'weights': True}   # the final pipeline's training variant (submission.FINAL)
QUIET = lambda s: None

HAVE_PFF = pff.available()
HAVE_DATA = atd.have_data() and not os.environ.get('MTAH_SKIP_TARGET')
params_path = 'data/models/degrade_params.json'
DEGRADE_PARAMS = json.load(open(params_path)) if os.path.isfile(params_path) else {}
print('PFF data present:', HAVE_PFF, '| BAR-ATH data present:', HAVE_DATA,
      '| calibrated noise:', bool(DEGRADE_PARAMS))

## 1. Convert PFF to match containers

One pass over every `.jsonl.bz2` (~10–20 min the first time, cached as `data/external/pff/npz/`).
PFF runs at ~30 fps; positions are linearly interpolated onto our 25 fps grid, labels taken from
the latest source frame. Only periods 1–2 are kept.

In [ ]:
if HAVE_PFF:
    paths = pff.convert_all(log=print)
    LOADERS = pff.loaders()
    FOLDS = min(FOLDS, len(LOADERS))
    print(len(LOADERS), 'PFF matches ready')
else:
    LOADERS = []
    print('skipped: no PFF data under', pff.RAW_DIR)

## 2. Does PFF fit? Inputs and possession definition

Same statistics as for IDSSE. The column that matters most is **possessions per alive minute**: the
public BAR-ATH window has ~4.2, IDSSE ~8.4. If PFF lands nearer 4–6 its event-based definition is
closer to Metrica's. (The public row compares *definitions*; nothing is fitted to it.)

In [ ]:
if HAVE_PFF:
    rows_in, rows_lab = [], []
    for load in LOADERS:
        m = load()
        rows_in.append(dict(match=m['name'], **dataset_fit.input_stats(m)))
        rows_lab.append(dict(match=m['name'], **dataset_fit.label_stats(m['label'])))
        del m
    pin, plab = pd.DataFrame(rows_in), pd.DataFrame(rows_lab)
    ilab = pd.DataFrame([dataset_fit.label_stats(m['label']) for m in idsse.load_all()])
    public = pm.label_codes(pm.read_label_csv(dataset_fit.PUBLIC_GT))
    keys = ['poss_per_alive_min', 'mean_poss_s', 'median_poss_s', 'lt1s', 'lt2s', 'share_D', 'mean_dead_s']
    summary = pd.DataFrame({'PFF (mean of %d)' % len(plab): plab[keys].mean(),
                            'IDSSE (mean of 7)': ilab[keys].mean(),
                            'BAR-ATH public 5 min': pd.Series(dataset_fit.label_stats(public))[keys]})
    display(summary.round(2))
    display(pin[['players_mean', 'ball_seen']].describe().round(3))

## 3. Noise

PFF's broadcast tracking already loses the ball on some frames; the harness only removes what is
needed to reach BAR-ATH's rate (`ev._top_up`). Check: raw PFF vs degraded PFF vs the target.

In [ ]:
if HAVE_PFF:
    m = LOADERS[0]()
    print('raw PFF     ', {k: round(v, 3) for k, v in degrade.atd_noise_stats(m).items()})
    d = degrade.degrade(m, seed=SEED, **(ev._top_up(m, DEGRADE_PARAMS) or {}))
    s = degrade.atd_noise_stats(d)
    print('degraded PFF', {k: round(v, 3) for k, v in s.items()})
    # verdict only: the target values are statistics of BAR-ATH and stay out of shared outputs
    target = dict(degrade.DEFAULTS, **DEGRADE_PARAMS)['ball_missing']
    print('ball-missing rate within 3 points of the BAR-ATH target:',
          'ok' if abs(s['ball_missing'] - target) < 0.03 else 'NO')
    del m, d

## 4. Evaluate on PFF (8 grouped folds): current decoder vs boundary-aware decoder

Each PFF match is predicted by a model that never saw it. The boundary decoder's grid contains the
current decoder (β = 0), so both are scored on identical folds. First run: features for 64 matches
+ 16 trainings, ~30–40 min; cached afterwards.

In [ ]:
def nested_best(res, allowed):
    grid = res['grid']; out = []
    for k in range(grid.shape[1]):
        others = [j for j in range(grid.shape[1]) if j != k]
        out.append(allowed[int(np.argmax(grid[allowed][:, others].mean(1)))])
    return out

if HAVE_PFF:
    t0 = time.time()
    pff_b = boundary.evaluate(LOADERS, FEATURE_FN, labels.raw, DEGRADE_PARAMS, seed=SEED,
                              n_folds=FOLDS, stride=PFF_STRIDE, swap=False, light=True,
                              neg_stride=PFF_STRIDE, bootstrap=200, log=QUIET, train_cfg=TRAIN_CFG)
    ev.report(pff_b['base'], 'PFF, current decoder')
    cfg = pff_b['pairs']; cur = [i for i, c in enumerate(cfg) if c[1] == 0]
    s_cur = [pff_b['grid'][g, k] for k, g in enumerate(nested_best(pff_b, cur))]
    s_new = [pff_b['grid'][g, k] for k, g in enumerate(nested_best(pff_b, list(range(len(cfg)))))]
    print('\nboundary decoder: mean per-match spellF1 {:.4f} vs current {:.4f}; better on {}/{} matches; '
          'best config (lambda, beta, dead) {}  [{:.0f}s]'.format(
          np.mean(s_new), np.mean(s_cur), int(np.sum(np.array(s_new) > np.array(s_cur))), len(s_cur),
          cfg[int(np.argmax(pff_b['grid'].mean(1)))], time.time() - t0))

## 5. The dataset matrix: what transfers?

Rows = what the model was trained on, columns = what it is scored on (always matches it never saw).
BAR-ATH is a third, unknown definition, so the row with the **best worst case** is the safest choice.

In [ ]:
if HAVE_PFF:
    t0 = time.time()
    kw = dict(feature_fn=FEATURE_FN, label_fn=labels.raw, degrade_params=DEGRADE_PARAMS, seed=SEED, log=QUIET)
    idsse_data = ev.prepare(idsse.load_all(), **kw)
    pff_data = ev.prepare(LOADERS, swap=False, light=True, **kw)
    cell = {}
    # trained on one dataset, scored on the other
    cell[('IDSSE', 'PFF')] = ev.cross(idsse_data, pff_data, log=QUIET, train_cfg=TRAIN_CFG)['pooled']['spell_f1_mean']
    cell[('PFF', 'IDSSE')] = ev.cross(pff_data, idsse_data, stride=PFF_STRIDE, log=QUIET,
                                      train_cfg=TRAIN_CFG)['pooled']['spell_f1_mean']
    # trained on the same dataset (out of fold) -- from earlier runs, cached
    cell[('IDSSE', 'IDSSE')] = ev.run(feature_fn=FEATURE_FN, degrade_params=DEGRADE_PARAMS, seed=SEED,
                                      bootstrap=0, log=QUIET, train_cfg=TRAIN_CFG)['pooled']['spell_f1_mean']
    cell[('PFF', 'PFF')] = pff_b['base']['pooled']['spell_f1_mean']
    # trained on both, held-out matches of one
    p1 = ev.oof_with_extra(idsse_data, pff_data, extra_stride=PFF_STRIDE, log=QUIET, train_cfg=TRAIN_CFG)
    cell[('both', 'IDSSE')] = ev.score(p1, idsse_data, bootstrap=0)['pooled']['spell_f1_mean']
    p2 = ev.oof_with_extra(pff_data, idsse_data, n_folds=FOLDS, stride=PFF_STRIDE, log=QUIET,
                           train_cfg=TRAIN_CFG)
    cell[('both', 'PFF')] = ev.score(p2, pff_data, bootstrap=0)['pooled']['spell_f1_mean']
    del idsse_data, pff_data
    matrix = pd.DataFrame({test: {train: cell[(train, test)] for train in ('IDSSE', 'PFF', 'both')}
                           for test in ('IDSSE', 'PFF')})
    matrix['worst case'] = matrix.min(1)
    display(matrix.round(4))
    TRAIN_ON = matrix['worst case'].idxmax()
    print('-> train the submission on:', TRAIN_ON, ' [{:.0f}s]'.format(time.time() - t0))

## 6. Final model and BAR-ATH submission

Same code path as the CLI (`submission.build`) with the final configuration
(`submission.FINAL`: v1 features, hand-over weights, boundary decoder (8, 2, 32), kickoff rule).
The only switch this notebook decides is whether PFF joins the training set (§5's worst case).

In [ ]:
if HAVE_PFF:
    WITH_PFF = TRAIN_ON in ('PFF', 'both')
    if TRAIN_ON == 'PFF':
        print('note: PFF-only training is not wired into submission.py; using IDSSE + PFF')
    best_cfg_pff = cfg[int(np.argmax(pff_b['grid'].mean(1)))]
    print('boundary decoder on PFF: best config there', best_cfg_pff,
          '| final uses', submission.FINAL['boundary'], '(chosen on IDSSE, 7/7 matches)')
    if HAVE_DATA:
        t0 = time.time()
        codes, digest = submission.build(TEAM, with_pff=WITH_PFF, log=QUIET)
        path = os.path.join('evaluation', 'results', TEAM + '.csv')
        print(submission.summary(codes)); print(path, 'sha256', digest, '[{:.0f}s]'.format(time.time() - t0))

## 7. Checks (alignment only — not a selection signal)

In [ ]:
if HAVE_PFF and HAVE_DATA:
    env = {**os.environ, 'PYTHONIOENCODING': 'utf-8'}
    for args in (['evaluation/validate_submission.py', path], ['evaluation/run_evaluation.py']):
        r = subprocess.run([sys.executable, *args], capture_output=True, text=True, encoding='utf-8', env=env)
        print(r.stdout, r.stderr)

## 8. Reproduce from the command line

In [ ]:
if HAVE_PFF:
    print(' '.join(['python', '-m', 'solution.submission', '--team', TEAM] + (['--with-pff'] if WITH_PFF else [])))